# Research: exploration → recorded experiment
Use a kernel with this project installed (`uv sync`). All execution is synchronous. No background jobs survive kernel exit.

In [ ]:
import torch

from core import analysis, groups, plotting, runner, tuning
from core.config import load
from core.training import Evaluation, evaluate
from methods import linear

## Explore entirely in memory
No registration or output directory is required.

In [ ]:
x = torch.randn(8, 3)
y = (x[:, 0] > 0).long()
model = linear.model()
values, predictions, targets = evaluate(
    Evaluation(model, [(x, y)], linear.batch, linear.metrics)
)
values

## Explicit preparation
Run the preparation cell only once. If the prepared example data already exists, skip that cell yourself; preparation intentionally refuses to overwrite it.

In [ ]:
config = load("configs/linear.toml")
config.train.epochs = 2

In [ ]:
runner.prepare(config)

## Full entry point (same as CLI)
An omitted ID generates a new run. Resume reuses that identity and preserves prior launch records. Test training settings are resolved from the selected checkpoint's launch automatically, even when the input below is the initial config.

In [ ]:
run_id = runner.execute(config)
config.train.epochs = 3
runner.execute(config, id=run_id, resume=True)
original = load(f"experiments/runs/{run_id}/config.toml")
test_id = runner.test(original, source=run_id)
analysis.detail(test_id)

## Search, export, explicit multi-seed training and testing
Trials use validation only. Export does not train or copy weights.

In [ ]:
study_id = tuning.tune(config, trials=2)
selected_file = tuning.export(
    config, study_id, f"experiments/reports/selected-{study_id}.toml"
)
selected = load(selected_file)
group_id = groups.create([selected], [1, 2])
groups.run_group(selected, group_id)
members = analysis.detail(group_id, folder="groups")["members"]
evaluations = groups.test_members(selected, group_id, members)
analysis.summary(evaluations, "accuracy")

## Read-only comparison and redraw
Figures can be restyled without rerunning any model. Dataset-name or protocol mismatches are rejected for aggregate statistics, but visibly marked in comparison tables. Default runs save validation-selected metrics; custom runs save their returned final metrics without requiring validation logs. Curves still need process records.

In [ ]:
analysis.table(
    evaluations, "accuracy", destination=f"experiments/reports/{group_id}.csv"
)
fig = plotting.curves(
    members, "accuracy", split="validation", title="Validation learning curves"
)
plotting.save_figure(fig, f"experiments/reports/{group_id}.png")
fig

## Final metrics and confusion matrix
Multi-seed bars use mean ± sample SD. Confusion matrices need class labels; converting logits is explicit.

In [ ]:
fig = plotting.bars([evaluations], "accuracy", labels=["Selected method"])
fig

In [ ]:
fig = plotting.classification(targets.numpy(), predictions.argmax(1).numpy())
fig

## Customize a figure
Pass native style settings when drawing, use explicit parameters for size, then adjust the returned axes or artists. This redraws the same saved results without rerunning experiments.

In [ ]:
import matplotlib.pyplot as plt

style = {"axes.prop_cycle": plt.cycler(color=["#0072B2"])}
fig = plotting.bars(
    [evaluations],
    "accuracy",
    labels=["Selected method"],
    style=style,
    figsize=(4, 3),
)
ax = fig.axes[0]
ax.set_ylabel("Accuracy")
ax.patches[0].set_hatch("//")
# plotting.save_figure(fig, f"experiments/reports/{group_id}-custom.pdf", style=style)
fig

See `paper_figures.ipynb` for independent custom-style and tueplots sizing examples.

For the variable-length dictionary/custom-loop example, use `configs/ragged.toml` with the same runner functions. See README for independent CLI/Slurm execution and explicit failed-member handling.